# Минутная стрелка: признаки важнее сложности модели

Учебный пример к **Lect04_MV.pptx, слайды 9–21**. Все модели реализованы средствами **scikit-learn**.

В презентации решается задача **восстановления минут по координатам конца стрелки**:
$(x,y)\mapsto \phi$. Мы последовательно попробуем одну координату, более сложные модели,
обе координаты и, наконец, новый признак — угол.

**Обозначения:** `x` и `y` — координатные признаки (см); `phi` — целевая
минутная компонента времени (мин), `phi_true` — наблюдения, `phi_predicted` — прогнозы.
Угол в радианах обозначаем `theta`: $\theta = 2\pi\phi / 60$.

Это воспроизведение постановки и учебного эксперимента, а не точных чисел со слайдов:
исходная выборка и настройки обучения в презентации не приложены. Здесь данные
генерируются с фиксированным seed. Для упрощения все модели обучаются и оцениваются
на одной и той же полной выборке, без выделения тестовой части.

**Ограничение:** метрики характеризуют подгонку к обучающим данным и могут быть
оптимистичными. Они не оценивают качество на новых наблюдениях и не позволяют
проверить переобучение. Для такой проверки нужна независимая тестовая выборка
или кросс-валидация.

Фотографии часов не используются: координаты считаются уже измеренными.

**Запуск:** выполните все ячейки сверху вниз (`Kernel → Restart Kernel and Run All Cells`).
Интернет и файл презентации для выполнения не нужны. Если библиотек нет, раскомментируйте
следующую ячейку установки, выполните её и перезапустите ядро.

In [ ]:
# %pip install numpy pandas matplotlib scikit-learn threadpoolctl

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from IPython.display import display
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures, StandardScaler
from threadpoolctl import threadpool_limits

SEED = 42
plt.rcParams.update({
    'figure.figsize': (10, 4), 'figure.dpi': 150, 'savefig.dpi': 150,
    'font.size': 11, 'axes.grid': True,
})
# PNG без дополнительного удвоения разрешения режимом retina.
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('png')
print(f'Python {sys.version.split()[0]} | NumPy {np.__version__} | sklearn {sklearn.__version__}')

## 1. Синтетические наблюдения циферблата

Центр часов — $(0,0)$, ось $x$ направлена вправо, ось $y$ — вверх.
Минутная стрелка длиной $R=6$ см поворачивается по часовой стрелке от вертикали:

$$\theta=\frac{2\pi \phi}{60},\qquad x=R\sin\theta,\qquad y=R\cos\theta.$$

Проверка соглашения: 0 минут → $(0,R)$; 15 → $(R,0)$; 30 → $(0,-R)$; 45 → $(-R,0)$.
К каждой координате добавляем независимый гауссовский шум измерения.

В основном эксперименте $\phi\in[0.5,59.5)$: небольшой зазор около 0/60 позволяет
сначала изучить неоднозначность одной координаты отдельно от разрыва минутной шкалы.
Это ограничение выборки, а не способ устранить цикличность. Переход через 0/60
исследуем отдельно в разделе 5. Минуты непрерывные, задача является регрессией.

In [ ]:
N_SAMPLES = 1800
RADIUS = 6.0             # см
COORD_NOISE = 0.08       # стандартное отклонение ошибки каждой координаты, см
MINUTE_MARGIN = 0.5      # можно установить 0 и изучить переход через 0/60


def ideal_coordinates(phi, radius=RADIUS):
    theta = 2 * np.pi * np.asarray(phi, dtype=float).reshape(-1) / 60
    return radius * np.column_stack([np.sin(theta), np.cos(theta)])


rng = np.random.default_rng(SEED)
phi_true = rng.uniform(MINUTE_MARGIN, 60 - MINUTE_MARGIN, N_SAMPLES)
xy_clean = ideal_coordinates(phi_true)
xy = xy_clean + rng.normal(0, COORD_NOISE, size=xy_clean.shape)

print(f'Обучение и оценка на одних данных: {len(phi_true)} наблюдений')
display(pd.DataFrame({'x, см': xy[:, 0], 'y, см': xy[:, 1], 'phi_true, мин': phi_true}).head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), layout='constrained')
points = axes[0].scatter(xy[:, 0], xy[:, 1], c=phi_true, cmap='viridis', s=8, alpha=0.7)
axes[0].set(xlabel='x, см', ylabel='y, см', title='Конец минутной стрелки', aspect='equal')
for minute, label in [(0, '0 / 60'), (15, '15'), (30, '30'), (45, '45')]:
    px, py = ideal_coordinates([minute], radius=RADIUS + 0.7)[0]
    axes[0].text(px, py, label, ha='center', va='center')
axes[0].set_xlim(-7.5, 7.5)
axes[0].set_ylim(-7.5, 7.5)
fig.colorbar(points, ax=axes[0], label='phi_true, мин', shrink=0.75)
for j, name in enumerate(['x', 'y']):
    axes[j + 1].scatter(xy[:, j], phi_true, s=7, alpha=0.35)
    axes[j + 1].set(xlabel=f'{name}, см', ylabel='phi_true, мин', title=f'Зависимость phi от {name}')
plt.show()

## 2. Почему одной координаты недостаточно?

Одинаковое значение $x$ может соответствовать разным минутам: например, 5 и 25.
Для $y$ такой пример — 10 и 50 минут. Шум не является причиной неоднозначности:
она существует и при идеальных измерениях.

При квадратичной функции потерь регрессия стремится к условному среднему
$\mathbb{E}[\phi\mid x]$ или $\mathbb{E}[\phi\mid y]$. Поэтому предсказание может оказаться
между двумя ветвями зависимости. Ни полином, ни нейросеть не восстановят
информацию о ветви, если её нет во входных данных.

In [ ]:
ambiguous_phi_true = np.array([5., 25., 10., 50.])
ambiguous_xy = ideal_coordinates(ambiguous_phi_true)
display(pd.DataFrame({
    'phi_true, мин': ambiguous_phi_true,
    'x, см': ambiguous_xy[:, 0],
    'y, см': ambiguous_xy[:, 1],
}).round(3))

## 3. Усложняем модель с одной координатой

Сравним линейную регрессию, полином степени 6
и многослойный перцептрон `MLPRegressor`.

`Pipeline` объединяет преобразования признаков и модель; они обучаются на всей выборке. Для MLP
масштабируем также целевую переменную через `TransformedTargetRegressor`:
метод `predict` автоматически возвращает результат в минутах.
Архитектура и настройки заданы заранее. Все метрики ниже рассчитаны на данных обучения.
Ограничение числа вычислительных потоков уменьшает накладные расходы на этой маленькой задаче.

In [ ]:
def make_mlp():
    return TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(32, 16), activation='tanh', solver='lbfgs',
                alpha=0.1, max_iter=2500, max_fun=50000, random_state=SEED,
            ),
        ),
        transformer=StandardScaler(),
    )


def circular_error(phi_true, phi_predicted):
    # Кратчайшая знаковая разность на окружности с периодом 60 минут.
    return (np.asarray(phi_predicted) - np.asarray(phi_true) + 30) % 60 - 30


def metrics_row(name, phi_true, phi_predicted):
    return {
        'Модель': name,
        'RMSE, мин': np.sqrt(mean_squared_error(phi_true, phi_predicted)),
        'MAE, мин': mean_absolute_error(phi_true, phi_predicted),
        'R²': r2_score(phi_true, phi_predicted),
        'Циклическая MAE, мин': np.mean(np.abs(circular_error(phi_true, phi_predicted))),
    }


one_coordinate_models = {}
rows = []
for column, coordinate in enumerate(['x', 'y']):
    models = {
        'Линейная': LinearRegression(),
        'Полином 6': make_pipeline(
            StandardScaler(), PolynomialFeatures(degree=6, include_bias=False),
            LinearRegression(),
        ),
        'MLP': make_mlp(),
    }
    # Срез column:column+1 сохраняет двумерную форму (n_samples, 1).
    X = xy[:, column:column + 1]
    for name, model in models.items():
        with threadpool_limits(limits=1):
            model.fit(X, phi_true)
        one_coordinate_models[(coordinate, name)] = model
        rows.append(metrics_row(f'{name}: {coordinate}', phi_true, model.predict(X)))

# Единый порядок для таблиц и диаграммы: модели одного типа стоят рядом.
model_order = [
    'Линейная: x', 'Линейная: y', 'Линейная: (x, y)',
    'Полином 6: x', 'Полином 6: y',
    'MLP: x', 'MLP: y', 'MLP: (x, y)',
]
one_coordinate_results = pd.DataFrame(rows).set_index('Модель')
one_coordinate_order = [name for name in model_order if name in one_coordinate_results.index]
display(one_coordinate_results.loc[one_coordinate_order].round(3))

### Каждая модель: зависимость от признака и прогноз на циферблате

Слева показаны наблюдаемые минуты `phi_true` в зависимости от единственного
признака `x` или `y` и красная кривая прогноза модели `phi_predicted`.
Справа точки расположены на циферблате согласно `phi_predicted`;
цвет на обоих графиках обозначает наблюдаемые минуты `phi_true`.
При точном прогнозе цвет соответствует минутной отметке на циферблате.
Прогнозы на циферблате отображаются по модулю 60, а RMSE рассчитывается
по исходным значениям. Все наблюдения используются для обучения и оценки.

In [ ]:
def plot_clock_predictions(fig, ax, phi_true, phi_predicted):
    colors = {'c': phi_true, 'cmap': 'viridis', 'vmin': 0, 'vmax': 60}
    circle = ideal_coordinates(np.linspace(0, 60, 361))
    ax.plot(circle[:, 0], circle[:, 1], color='gray', lw=1, zorder=1)
    predicted_xy = ideal_coordinates(np.asarray(phi_predicted) % 60)
    points = ax.scatter(predicted_xy[:, 0], predicted_xy[:, 1], s=22,
                        alpha=0.65, zorder=3, **colors)
    for minute in range(60):
        inner_radius = RADIUS * (0.94 if minute % 5 == 0 else 0.97)
        tick = np.vstack([
            ideal_coordinates([minute], radius=inner_radius),
            ideal_coordinates([minute]),
        ])
        ax.plot(tick[:, 0], tick[:, 1], color='gray', lw=1)
        if minute % 5 == 0:
            px, py = ideal_coordinates([minute], radius=RADIUS * 1.12)[0]
            ax.text(px, py, '0 / 60' if minute == 0 else str(minute),
                    ha='center', va='center')
    ax.scatter(0, 0, color='gray', s=12)
    ax.set(
        xlabel='x, см', ylabel='y, см', title='Положения phi_predicted на циферблате',
        xlim=(-RADIUS * 1.25, RADIUS * 1.25),
        ylim=(-RADIUS * 1.25, RADIUS * 1.25), aspect='equal',
    )
    fig.colorbar(points, ax=ax, label='Наблюдения (phi_true), мин', shrink=0.8)


def plot_one_coordinate_model(coordinate, model_name):
    column = {'x': 0, 'y': 1}[coordinate]
    model = one_coordinate_models[(coordinate, model_name)]
    feature = xy[:, column:column + 1]
    phi_predicted = model.predict(feature)
    rmse = np.sqrt(mean_squared_error(phi_true, phi_predicted))
    grid = np.linspace(feature.min(), feature.max(), 500).reshape(-1, 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 6), layout='constrained')
    fig.suptitle(f'{model_name}: {coordinate} — RMSE на обучении = {rmse:.3f} мин')
    ax = axes[0]
    ax.scatter(feature[:, 0], phi_true, c=phi_true, cmap='viridis', vmin=0, vmax=60,
               s=16, alpha=0.6, label='Наблюдения (phi_true)')
    ax.plot(grid[:, 0], model.predict(grid), color='crimson', lw=2,
            label='Модель (phi_predicted)')
    ax.set(xlabel=f'Признак {coordinate}, см', ylabel='phi, мин',
           title=f'Наблюдения и модель в зависимости от {coordinate}',
           xlim=(-RADIUS - 0.3, RADIUS + 0.3), ylim=(-5, 65))
    ax.set_box_aspect(1)
    ax.legend()
    plot_clock_predictions(fig, axes[1], phi_true, phi_predicted)
    plt.show()

In [ ]:
plot_one_coordinate_model('x', 'Линейная')

In [ ]:
plot_one_coordinate_model('y', 'Линейная')

In [ ]:
plot_one_coordinate_model('x', 'Полином 6')

In [ ]:
plot_one_coordinate_model('y', 'Полином 6')

In [ ]:
plot_one_coordinate_model('x', 'MLP')

In [ ]:
plot_one_coordinate_model('y', 'MLP')

## 4. Обе координаты без явного вычисления угла

Передадим нейросети сразу $(x,y)$, оставив ту же архитектуру и настройки.
Для сравнения обучим и линейную модель на двух координатах: полная информация
сама по себе ещё не делает зависимость линейной.

На слайде 21 указан RMSE 0.28 мин. Здесь получатся другие значения из-за другой
выборки, уровня шума, архитектуры и способа оценки. Сравнивать следует прежде всего
изменение качества подгонки на одной и той же обучающей выборке.

In [ ]:
linear_xy = LinearRegression().fit(xy, phi_true)
mlp_xy = make_mlp()
with threadpool_limits(limits=1):
    mlp_xy.fit(xy, phi_true)

phi_predicted_linear_xy = linear_xy.predict(xy)
phi_predicted_mlp_xy = mlp_xy.predict(xy)
rows.append(metrics_row('Линейная: (x, y)', phi_true, phi_predicted_linear_xy))
rows.append(metrics_row('MLP: (x, y)', phi_true, phi_predicted_mlp_xy))
results_xy = pd.DataFrame(rows).set_index('Модель').loc[model_order]
display(results_xy.round(3))

### Прогноз и наблюдения: диаграмма рассеяния и циферблат

Слева по горизонтали отложены наблюдаемые минуты (`phi_true`), по вертикали — прогноз
(`phi_predicted`). Целевая переменная `phi` измеряется в минутах, а `x` и `y` — координаты
конца стрелки в сантиметрах. Пунктир — идеальный прогноз; масштабы одинаковы для всех моделей.

Справа каждый прогноз показан как положение конца минутной стрелки на циферблате:
координаты точки вычислены из `phi_predicted`. Цвет точки на обоих графиках обозначает
**наблюдаемое время `phi_true`**, поэтому можно увидеть, куда модель перемещает наблюдения.
При точном прогнозе цвет соответствует минутной отметке в этом месте циферблата.
Прогнозы отображаются по модулю 60; обычный RMSE рассчитывается по исходным значениям.
Все точки относятся к данным обучения.

In [ ]:
def plot_predictions(phi_true, phi_predicted, title):
    rmse = np.sqrt(mean_squared_error(phi_true, phi_predicted))
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), layout='constrained')
    fig.suptitle(f'{title} — RMSE на обучении = {rmse:.3f} мин')
    colors = {'c': phi_true, 'cmap': 'viridis', 'vmin': 0, 'vmax': 60}

    ax = axes[0]
    ax.scatter(phi_true, phi_predicted, s=16, alpha=0.6, **colors)
    ax.plot([-5, 65], [-5, 65], 'k--', lw=1, label='Идеальный прогноз')
    ax.set(
        xlabel='Наблюдения (phi_true), мин', ylabel='Прогноз (phi_predicted), мин',
        title='Прогноз и наблюдения',
        xlim=(-5, 65), ylim=(-5, 65), aspect='equal',
    )
    ax.legend()

    plot_clock_predictions(fig, axes[1], phi_true, phi_predicted)
    plt.show()


plot_predictions(phi_true, phi_predicted_linear_xy, 'Линейная регрессия: (x, y)')

In [ ]:
plot_predictions(phi_true, phi_predicted_mlp_xy, 'MLP: (x, y)')

### Сравнение моделей на исходных координатах

Модели сгруппированы по типу: линейная регрессия, полином степени 6, MLP.
Порядок совпадает с таблицей выше; цвет обозначает тип модели.
Чем меньше RMSE на обучении, тем точнее модель описывает имеющиеся наблюдения.
Качество на новых данных здесь не проверяется.

Здесь сравниваем модели на одной или обеих исходных координатах. Модель с новым угловым признаком добавим в заключительном разделе.

In [ ]:
def plot_model_comparison(results, title):
    type_colors = {'Линейная': 'steelblue', 'Полином 6': 'darkorange', 'MLP': 'seagreen'}
    bar_colors = [type_colors[name.split(':')[0]] for name in results.index]
    fig, ax = plt.subplots(figsize=(10, 6), layout='constrained')
    results['RMSE, мин'].plot.barh(ax=ax, color=bar_colors)
    ax.invert_yaxis()  # Порядок сверху вниз совпадает с таблицей.
    ax.set(xlabel='RMSE на обучении, мин', ylabel='', title=title)
    ax.set_xlim(0, results['RMSE, мин'].max() * 1.15)
    ax.grid(axis='y', visible=False)
    ax.bar_label(ax.containers[0], fmt='%.3f', padding=4)
    plt.show()


plot_model_comparison(results_xy, 'Модели на исходных координатах: без углового признака')

## 5. Особенность времени: 59.9 и 0.1 минуты находятся рядом

Обычные MSE, RMSE и MAE рассматривают минуты как числа на прямой. Если истинное
время 59.9, а прогноз 0.1, абсолютная ошибка равна 59.8 мин, хотя стрелка
ошиблась всего на 0.2 мин. Для положения стрелки естественна циклическая ошибка:

$$e_{\mathrm{circ}}=((\hat \phi-\phi+30)\bmod60)-30.$$

В таблице выше RMSE оставлен для сопоставления с презентацией, а циклическая MAE
показывает ошибку положения на циферблате. При необходимости предсказывать именно
положение полезно обучать модель на паре $(\sin\theta,\cos\theta)$ вместо скалярных минут.

Ниже — отдельная иллюстрация поведения метрик около границы на шумных координатах.
Используем только точную геометрическую формулу, без обучения и оценки ML-моделей.
Даже точная формула может иметь большой обычный RMSE из-за шума около границы.

In [ ]:
print('Обычная ошибка 59.9 → 0.1:', abs(0.1 - 59.9))
print('Циклическая ошибка:', abs(circular_error(59.9, 0.1)))

edge_rng = np.random.default_rng(SEED + 1)
edge_phi_true = np.r_[edge_rng.uniform(0, 0.25, 250), edge_rng.uniform(59.75, 60, 250)]
edge_xy = ideal_coordinates(edge_phi_true) + edge_rng.normal(0, COORD_NOISE, (500, 2))
edge_phi_predictions = {
    'Геометрическая формула': np.mod(np.arctan2(edge_xy[:, 0], edge_xy[:, 1]), 2 * np.pi) * 60 / (2 * np.pi),
}
display(pd.DataFrame([
    metrics_row(name, edge_phi_true, phi_predicted) for name, phi_predicted in edge_phi_predictions.items()
]).set_index('Модель').round(3))

## 6. Порождаем новый признак: угол

Из **обеих** координат восстановим угол от вертикали по часовой стрелке:

$$\theta=\operatorname{atan2}(x,y)\bmod 2\pi,\qquad \phi\approx k\theta+b.$$

У `np.arctan2` первый аргумент обычно обозначают как $y$, а второй — как $x$.
Здесь намеренно передаём **`arctan2(x, y)`**, поскольку отсчитываем угол от верхней
вертикали. Эта формула однозначно учитывает четверть и согласована с выбранными осями;
при переносе формулы через `arccos` со слайда 18 нужно отдельно проверять оси и ветви.
В точке $(0,0)$ направление не определено: такие измерения нельзя интерпретировать как время.

Теперь достаточно `LinearRegression`: в идеальном случае $k=60/(2\pi)$, $b=0$.
Угол не создаёт недостающую информацию из одной координаты — при его вычислении
используются обе координаты и знание геометрии часов. Целевые минуты при преобразовании не используются.

In [ ]:
def clock_angle(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape[1] != 2:
        raise ValueError('Ожидается массив координат формы (n_samples, 2).')
    if np.any(np.hypot(X[:, 0], X[:, 1]) < 1e-12):
        raise ValueError('В центре циферблата угол не определён.')
    return np.mod(np.arctan2(X[:, 0], X[:, 1]), 2 * np.pi).reshape(-1, 1)


angle_model = make_pipeline(
    FunctionTransformer(clock_angle, validate=True), LinearRegression()
)
angle_model.fit(xy, phi_true)
phi_predicted_angle = angle_model.predict(xy)
rows.append(metrics_row('Линейная: угол из (x, y)', phi_true, phi_predicted_angle))

angle_regression = angle_model.named_steps['linearregression']
print(f'k = {angle_regression.coef_[0]:.4f}; теория: {60 / (2 * np.pi):.4f}')
print(f'b = {angle_regression.intercept_:.4f} мин')

theta_grid = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(clock_angle(xy)[:, 0], phi_true, s=12, alpha=0.4, label='Данные обучения')
ax.plot(theta_grid[:, 0], angle_regression.predict(theta_grid), color='crimson', label='Линейная модель')
ax.set(xlabel='Угол theta от вертикали по часовой стрелке, рад', ylabel='phi_true, мин', title='Переход к углу')
ax.legend()
plt.show()

In [ ]:
plot_predictions(phi_true, phi_predicted_angle, 'Линейная регрессия: угол из (x, y)')

# Угловая модель остаётся рядом с остальными линейными моделями.
final_model_order = model_order.copy()
final_model_order.insert(3, 'Линейная: угол из (x, y)')
results = pd.DataFrame(rows).set_index('Модель').loc[final_model_order]
display(results.round(3))

## Выводы и задания для самостоятельной работы

1. Одна координата не определяет минуты однозначно. Сложность модели не заменяет недостающий признак.
2. Нейросеть на двух координатах может выучить зависимость без явной формулы угла.
3. Выбор метрики должен учитывать цикличность времени, особенно около 0/60.
4. Угол из двух координат превращает зависимость в почти линейную. Это пример использования предметных знаний.
5. Оценка на данных обучения показывает качество подгонки; способность обобщать на новые данные здесь не проверяется.

**Попробуйте:** увеличьте `COORD_NOISE`; установите `MINUTE_MARGIN = 0`;
уменьшите `N_SAMPLES`. Каждый раз выполняйте ноутбук целиком и объясняйте изменения
обычного RMSE и циклической ошибки. В качестве отдельного расширения добавьте
разделение на обучающую и тестовую выборки и сравните ошибки на них.

Связь с гидрометеорологией: час суток, день года и направление ветра тоже являются
циклическими величинами. Представление через синус и косинус сохраняет близость
значений по разные стороны границы периода.

### Итоговое сравнение: добавляем модель с угловым признаком

Теперь на диаграмме представлены все модели, включая линейную регрессию на угле.
Группировка по типу модели и цвета сохранены. Метрики рассчитаны на данных обучения.

In [ ]:
plot_model_comparison(results, 'Все модели: включая новый угловой признак')